# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Data Wrangling for the NLP project

## Overview

The data is from IPUMS 2024.

I divided the features into two main sets:
- The **context features**, used to construct the demographic persona in the LLM prompt. These can be continuous or categorical.
- The **inference features**, which the LLM predicts and we evaluate against ACS ground truth. These are **always** categorical.

A variable can serve as context in one experiment and as the hidden inference target in another. The role assignments below indicate each variable's primary intended use.

## Technical Variables

These are not features. They are used for weighting and survey design but never appear in prompts.

- `PERWT`, continuous, float
  Person-level sampling weight. Used to compute population-representative ground-truth distributions from the ACS microdata. Essential for all evaluation metrics (KL divergence, chi-squared, marginal comparisons).

- `YEAR`, categorical, int
  Census/survey year. Used to select the relevant ACS sample.


## Context Features

These variables define the demographic persona provided to the LLM in the prompt. All features **do not** have to be used but I reference and prepare them here in case we need them later.

### Core Demographics

- `AGE`, continuous, int
  Age of the individual in years.

- `SEX`, categorical, int (2 categories)
  Birth sex of the individual: male or female.

- `RACE`, categorical, int (9 categories in IPUMS)
  Racial identification of the individual. Major categories include White, Black/African American, American Indian, Chinese, Japanese, Other Asian/Pacific Islander, Other race, Two major races, Three or more major races. Need collapsing to avoid thin cells.

- `HISPAN`, categorical, int (5 categories)
  Hispanic origin: Not Hispanic, Mexican, Puerto Rican, Cuban, Other. Can be combined with RACE to construct a joint race/ethnicity variable. Consider collapsing to a binary Hispanic/non-Hispanic indicator depending on cell sizes.

- `EDUC`, categorical, int (11 categories)
  Educational attainment. Ranges from no schooling to 5+ years of college. Consider collapsing into a smaller set (e.g., less than high school, high school, some college, bachelor's, graduate) for cleaner prompt language and evaluation cells.

- `STATEFIP`, categorical, int (51 categories)
  State FIPS code identifying the respondent's state of residence. Used both as context in the prompt (state name) and as the geographic unit for behavioral validation against election returns.

- `MARST`, categorical, int (6 categories)
  Marital status: married (spouse present), married (spouse absent), separated, divorced, widowed, never married. Candidate for both context and inference roles.

- `HCOVANY`, categorical, int (2 categories)
  Health insurance coverage: has any coverage vs. no coverage. Binary and very representative of social status.

### Income

Need to choose one of the three income variables as the primary prompt input to avoid redundancy.

- `INCTOT`, continuous, int
  Total personal income from all sources. Will be binned into brackets for prompt text (e.g., "$25k–$50k"). Denominated in nominal dollars.

- `HHINCOME`, continuous, int
  Total household income. Alternative to INCTOT; more intuitive for persona prompts but measured at the household level.

- `INCWAGE`, continuous, int
  Wage and salary income. Narrower than INCTOT (excludes transfers, investment income, etc.). Connects most directly to labor market outcomes.

### Work

- `EMPSTAT`, categorical, int (3 categories)
  Employment status: employed, unemployed, not in labor force. Cleanest candidate for primary hidden feature — low cardinality, strong demographic gradients, politically relevant, and well-suited for chi-squared evaluation.

- `CLASSWKR`, categorical, int
  Refers to the current or last job.
  - 1: self-employed.
  - 2: Works for wages


- `UHRSWORK`, continuous, int
  Used to evaluate the hourly wage.

- `TRANWORK`, categorical, int (variable cardinality)
  Means of transportation to work: car, public transit, bicycle, walked, worked from home, other. Only defined for workers. Strong geographic and income gradients — tests whether the LLM captures conditional behavioral structure beyond core demographics.

### Household and Family Structure

- `NCHILD`, categorical/continuous, int
  Number of own children present in the household. Can be used as continuous context or collapsed into a binary (has children / no children). Typically a strong predictor of political attitudes and time-use behavior.

- `OWNERSHP`, categorical, int (3 categories)
  Housing tenure: owned, rented, or N/A (group quarters). Captures a wealth/stability dimension that income alone does not. Binary (owner vs. renter) after dropping group quarters.

### Nativity and Language

- `SPEAKENG`, categorical, int (6 categories)
  English-speaking ability: does not speak English, yes (speaks only English), yes (speaks very well), yes (speaks well), yes (but not well), N/A. Interacts with how the LLM conceptualizes the persona.

## Inference Feature Candidates

These categorical variables are the primary candidates for the hidden feature that the LLM predicts.

- `RACE`&`HISPAN` - Interesting to investigate if the LLM prejudices certain races based on individual characteristics.

- `EMPSTAT` - Interesting to see if based on a persona profile, the LLM predicts different employment status.

## Design Notes

1. **Income binning.** The three income variables (`INCTOT`, `HHINCOME`, `INCWAGE`) are continuous in the ACS. For prompting, they will be converted to labeled brackets. The bracket scheme should be documented and held constant across experiments, since LLM priors may be sensitive to phrasing ("$50k–$75k" vs. "middle income").

2. **Cell sparsity.** Crossing 5+ categorical variables might produce thin cells.

3. Will filter for individuals with age 25-64 years old.

4. **Universe restrictions.** Some variables are only defined for subpopulations: `CLASSWKR` and `TRANWORK` require employment, `INCWAGE` requires labor force participation. This can "leak" information based on the target feature.

## Data Import

In [ ]:
import os
import pandas as pd

# Obtain an authorized ACS/IPUMS extract separately.
output = os.environ.get("ACS_DATA_PATH", "usa_00006.csv.gz")
df = pd.read_csv(output)


# Data Filtering

In [ ]:
def log_step(df, label):
    print(f"{label}")
    print(f"  Observations: {len(df):,}")
    print(f"  Weighted population: {df['PERWT'].sum():,.0f}")
    print("-" * 50)
    return df

# Initial state
log_step(df, "Initial dataset")

# === Sample Filters ===

df = df[(df["AGE"] >= 25) & (df["AGE"] <= 64)]
log_step(df, "After AGE filter (25–64)")

df = df[df["GQ"].isin([1, 2, 5])]
log_step(df, "After GQ filter (households only)")

df = df[df["INCTOT"] >= 0]
log_step(df, "After INCTOT filter (>= 0)")

df = df[df["HHINCOME"] != 9999999].copy()
log_step(df, "After HHINCOME filter (!= 9999999)")

# === Missing/Invalid Value Filters ===

df = df[df["SEX"] != 9]
log_step(df, "After SEX filter")

df = df[df["MARST"] != 9]
log_step(df, "After MARST filter")

df = df[df["HISPAN"] != 9]
log_step(df, "After HISPAN filter")

df = df[~df["SPEAKENG"].isin([0, 7, 8, 9])]
log_step(df, "After SPEAKENG filter")

df = df[df["EDUC"] != 99]
log_step(df, "After EDUC filter")

df = df[~df["EMPSTAT"].isin([0, 9])]
log_step(df, "After EMPSTAT filter")

df = df[df["OWNERSHP"] != 0]
log_step(df, "After OWNERSHP filter")

df = df[df["OWNERSHP"] != 0]
log_step(df, "After OWNERSHP filter")

df = df[~df["CLASSWKRD"].isin([0, 9])]
log_step(df, "After CLASSWKRD filter")

df.reset_index(drop=True, inplace=True)
# Final
print("\nFinal sample:")
print(f"Remaining observations: {len(df):,}")
print(f"Weighted population: {df['PERWT'].sum():,.0f}")

## Data cleaning


In [ ]:
df.drop(columns=["GQ", "PERNUM", "EDUCD", "EMPSTATD", "CLASSWKRD", "OWNERSHPD", "RACED", "HISPAND"], inplace=True)


In [ ]:
#State names

STATEFIP_map = {
    1: 'Alabama',
    2: 'Alaska',
    4: 'Arizona',
    5: 'Arkansas',
    6: 'California',
    8: 'Colorado',
    9: 'Connecticut',
    10: 'Delaware',
    11: 'District of Columbia',
    12: 'Florida',
    13: 'Georgia',
    15: 'Hawaii',
    16: 'Idaho',
    17: 'Illinois',
    18: 'Indiana',
    19: 'Iowa',
    20: 'Kansas',
    21: 'Kentucky',
    22: 'Louisiana',
    23: 'Maine',
    24: 'Maryland',
    25: 'Massachusetts',
    26: 'Michigan',
    27: 'Minnesota',
    28: 'Mississippi',
    29: 'Missouri',
    30: 'Montana',
    31: 'Nebraska',
    32: 'Nevada',
    33: 'New Hampshire',
    34: 'New Jersey',
    35: 'New Mexico',
    36: 'New York',
    37: 'North Carolina',
    38: 'North Dakota',
    39: 'Ohio',
    40: 'Oklahoma',
    41: 'Oregon',
    42: 'Pennsylvania',
    44: 'Rhode Island',
    45: 'South Carolina',
    46: 'South Dakota',
    47: 'Tennessee',
    48: 'Texas',
    49: 'Utah',
    50: 'Vermont',
    51: 'Virginia',
    53: 'Washington',
    54: 'West Virginia',
    55: 'Wisconsin',
    56: 'Wyoming',
    61: 'Maine-New Hampshire-Vermont',
    62: 'Massachusetts-Rhode Island',
    63: 'Minnesota-Iowa-Missouri-Kansas-Nebraska-S.Dakota-N.Dakota',
    64: 'Maryland-Delaware',
    65: 'Montana-Idaho-Wyoming',
    66: 'Utah-Nevada',
    67: 'Arizona-New Mexico',
    68: 'Alaska-Hawaii',
    72: 'Puerto Rico',
    97: 'Military/Mil. Reservation',
    99: 'State not identified'
}

df["STATE_NAME"] = df['STATEFIP'].map(STATEFIP_map)
df.drop(columns=["STATEFIP"], inplace=True)

In [ ]:
OWNERSHP_map = {
    1: 'Owned or being bought (loan)',
    2: 'Rented'
}

df["OWNERSHP_STATUS"] = df['OWNERSHP'].map(OWNERSHP_map)
df.drop(columns=["OWNERSHP"], inplace=True)


In [ ]:
SEX_map = {
    1: 'Male',
    2: 'Female'
}

df["SEX"] = df['SEX'].map(SEX_map)

In [ ]:
MARST_map = {
    1: 'Married, spouse present',
    2: 'Married, spouse absent',
    3: 'Separated',
    4: 'Divorced',
    5: 'Widowed',
    6: 'Single'
}

df["MARITAL_STATUS"] = df['MARST'].map(MARST_map)
df.drop(columns=["MARST"], inplace=True)

In [ ]:
import numpy as np

# 1. Filter the data: Keep only Race codes 1-6 OR any Hispanic identification
# This effectively DROPS 'Other race, nec' (7), 'Two major races' (8), and 'Three or more' (9)
# unless they identify as Hispanic.
df = df[df['HISPAN'].isin([1, 2, 3, 4]) | df['RACE'].isin([1, 2, 3, 4, 5, 6])].copy()

# 2. Define the mapping for the remaining Race categories
# Short definitions for likelihood biases
RACE_map = {
    1: 'White',         # White
    2: 'Black',         # Black/African American
    3: 'Indigenous',    # American Indian or Alaska Native
    4: 'Chinese',       # Chinese
    5: 'Japanese',      # Japanese
    6: 'Other',         # Other Asian or Pacific Islander
}

# 3. Create the unique column
# Logic: If Hispanic (1-4), label as 'Hispanic'.
# Otherwise, use the Race label (which is now strictly 1-6 due to the drop above).
conditions = [
    df['HISPAN'].isin([1, 2, 3, 4]),
    df['HISPAN'].isin([0, 9])
]

choices = [
    'Hispanic',
    df['RACE'].map(RACE_map)
]

df['RACE_ETHNICITY'] = np.select(conditions, choices, default='Other/Unknown')

# 4. Cleanup: Remove any remaining 'Other/Unknown' if necessary
# (e.g., cases where HISPAN was 9 and RACE was not in 1-6)
df = df[df['RACE_ETHNICITY'] != 'Other/Unknown']

df.drop(columns=["RACE", "HISPAN"], inplace=True)

print(df['RACE_ETHNICITY'].value_counts())

In [ ]:
SPEAKENG_map = {
    1: 'Does not speak English',
    2: 'Speaks English',
    3: 'Speaks English',
    4: 'Speaks English',
    5: 'Speaks English',
    6: 'Speaks English' # "Yes, but not well" - included in Speaks English
}

# 2. Create the new variable
# Note: Using 'SPEAKENG' as the column name (standard IPUMS),
# change to 'SPEAKENG_eng' if that is your specific column name.
df['ENGLISH_STATUS'] = df['SPEAKENG'].map(SPEAKENG_map)
df.drop(columns=["SPEAKENG"], inplace=True)

# 3. Verification
print(f"Value Counts: {df['ENGLISH_STATUS'].value_counts(dropna=False)}")

In [ ]:
HCOVANY_map = {
    1: 'No health insurance coverage',
    2: 'With health insurance coverage'
}

df['HEALTH_INSURANCE'] = df['HCOVANY'].map(HCOVANY_map)
df.drop(columns=["HCOVANY"], inplace=True)


In [ ]:
EMPSTAT_map = {
    1: 'Employed',
    2: 'Unemployed',
    3: 'Not in labor force'
}

# Note: Since 0 (N/A) is not in this map, it will automatically become NaN.
df['EMPLOYMENT_STATUS'] = df['EMPSTAT'].map(EMPSTAT_map)
df.drop(columns=["EMPSTAT"], inplace=True)


In [ ]:
EDUC_map = {
    0:  'No high school',     # No schooling
    1:  'No high school',     # Nursery school to grade 4
    2:  'No high school',     # Grade 5, 6, 7, or 8
    3:  'No high school',     # Grade 9
    4:  'No high school',     # Grade 10
    5:  'No high school',     # Grade 11
    6:  'High school',        # Grade 12
    7:  'Some college',       # 1 year of college
    8:  'Some college',       # 2 years of college
    9:  'Some college',       # 3 years of college
    10: 'Bachelors',          # 4 years of college (no apostrophe)
    11: 'Graduate degree',    # 5+ years of college
}

df['EDUCATION_LEVEL'] = df['EDUC'].map(EDUC_map)
df.drop(columns=["EDUC"], inplace=True)

In [ ]:
CLASSWKR_map = {
    1: 'Self-employed',
    2: 'Works for wages'
}

# Values 0 (N/A) and 9 (Unknown) will automatically result in NaN when using .map()
df['WORKER_CLASS'] = df['CLASSWKR'].map(CLASSWKR_map)
df.drop(columns=["CLASSWKR"], inplace=True)


In [ ]:
TRANWORK_map = {
    # Private Vehicle
    10: 'Private Vehicle', # Auto, truck, or van
    11: 'Private Vehicle', # Auto
    12: 'Private Vehicle', # Driver
    13: 'Private Vehicle', # Passenger
    14: 'Private Vehicle', # Truck
    15: 'Private Vehicle', # Van
    20: 'Private Vehicle', # Motorcycle

    # Public Transit
    31: 'Public Transit',  # Bus
    32: 'Public Transit',  # Bus or trolley
    33: 'Public Transit',  # Bus or streetcar
    34: 'Public Transit',  # Light rail/Subway
    35: 'Public Transit',  # Streetcar/Trolley
    36: 'Public Transit',  # Subway or elevated
    37: 'Public Transit',  # Long-distance/Commuter train
    39: 'Public Transit',  # Ferryboat

    # Active Transport
    50: 'Active Transport', # Bicycle
    60: 'Active Transport', # Walked only

    # Work from Home
    80: 'Worked at home',

    # Other / Taxi
    38: 'Other/Taxi',       # Taxicab or ride-hailing
    70: 'Other/Taxi'        # Other
}

# Values 00 (N/A) will automatically become NaN when using .map()
df['TRANSPORT_METHOD'] = df['TRANWORK'].map(TRANWORK_map)
df.drop(columns=["TRANWORK"], inplace=True)


In [ ]:
df.columns

In [ ]:
df.dropna(inplace=True)
# Final
print("\nFinal sample:")
print(f"Remaining observations: {len(df):,}")
print(f"Weighted population: {df['PERWT'].sum():,.0f}")

TBD:

- Group `AGE` into categories?


In [ ]:
import pandas as pd
import numpy as np

# Define the bins and labels for AGE
# Bins are right-inclusive by default in pd.cut, so:
# (17, 24] -> 18-24
# (24, 34] -> 25-34
# (34, 44] -> 35-44
# (44, 54] -> 45-54
# (54, 64] -> 55-64
# (64, inf] -> 65+
bins = [17, 24, 34, 44, 54, 64, np.inf]
labels = ['18–24', '25–34', '35–44', '45–54', '55–64', '65+']

# Create the AGE_BINS column
df['AGE_BINS'] = pd.cut(df['AGE'], bins=bins, labels=labels)

print("Age Bin Distribution:")
print(df['AGE_BINS'].value_counts())


In [ ]:
# Group income into categories

# Drop missing/NA sentinel codes and no-income/loss rows
df = df[
    (~df['INCTOT'].isin([9999999, 9999998])) &
    (df['INCTOT'] > 0)
].copy()

# Define bins and labels for positive income only
bins = [0, 14999, 29999, 49999, 74999, 149999, np.inf]

labels = [
    '$1–$14,999',
    '$15,000–$29,999',
    '$30,000–$49,999',
    '$50,000–$74,999',
    '$75,000–$149,999',
    '$150,000+',
]

# Create INCTOT_BINS
df['INCTOT_BINS'] = pd.cut(
    df['INCTOT'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print("Income Bin Distribution:")
print(df['INCTOT_BINS'].value_counts())

print("\nWeighted (PERWT) population shares:")
print(
    df.groupby('INCTOT_BINS', observed=True)['PERWT'].sum()
    / df['PERWT'].sum()
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# df.to_parquet("/content/drive/MyDrive/Aa_cours/PhD6/6_8610/NLP_project/data/demographic_data_clean.parquet", compression='snappy')

---

## OUTPUT DATA

### 1. Dataset Overview
* **Universe:** Individuals aged 25–64, living in households (excluding group quarters).
* **Filtering:** Rows with missing or invalid values for core demographics (Sex, Marital Status, Education, etc.) have been removed.


### 2. Technical Variables
These variables are used for internal processing and statistical weighting.

| Column | Type | Description |
| :--- | :--- | :--- |
| `PERWT` | Continuous | **Person Weight**: Indicates how many people in the US population the individual represents. |




### 3. Context Features (Demographic Persona)
These variables form the "Persona" provided to the LLM.

#### Core Identity
| Column | Type | Description / Levels |
| :--- | :--- | :--- |
| `AGE` | Continuous | Age of the individual in years. |
| `AGE_BINS` | Categorical | Age bracket. Levels: `18–24`, `25–34`, `35–44`, `45–54`, `55–64`, `65+` (Note: universe previously filtered to 25–64). |
| `SEX` | Categorical | `Male`, `Female` |
| `RACE_ETHNICITY` | Categorical | **Combined Race & Hispanic Origin**. Priority is given to Hispanic identity. If `HISPAN` is 1-4, value is `Hispanic`. Otherwise: `White`, `Black/African American`, `American Indian or Alaska Native`, `Chinese`, `Japanese`, `Other Asian or Pacific Islander`. |
| `MARITAL_STATUS` | Categorical | `Married, spouse present`, `Married, spouse absent`, `Separated`, `Divorced`, `Widowed`, `Single` |
| `STATE_NAME` | Categorical | Full name of the state of residence (51 categories, including DC). |
| `ENGLISH_STATUS` | Categorical | `Speaks English` (includes "well", "very well", "only"), `Does not speak English`. |

#### Socio-Economic & Household
| Column | Type | Description / Levels |
| :--- | :--- | :--- |
| `EDUCATION_LEVEL` | Categorical | Highest grade completed. Levels range from `No schooling` to `5+ years of college`. |
| `OWNERSHP_STATUS` | Categorical | `Owned or being bought (loan)`, `Rented`. |
| `NCHILD` | Continuous | Number of own children present in the household. |
| `HEALTH_INSURANCE`| Categorical | `With health insurance coverage`, `No health insurance coverage`. |


### 4. Economic & Work Features
These can serve as context or as inference targets (prediction tasks).

#### Income
| Column | Type | Description |
| :--- | :--- | :--- |
| `INCTOT` | Continuous | Total personal income from all sources (Filtered: >= 0). |
| `HHINCOME` | Continuous | Total household income. |
| `INCWAGE` | Continuous | Total wage and salary income. |

#### Labor & Commute
| Column | Type | Description / Levels |
| :--- | :--- | :--- |
| `EMPLOYMENT_STATUS`| Categorical | `Employed`, `Unemployed`, `Not in labor force`. |
| `WORKER_CLASS` | Categorical | `Self-employed`, `Works for wages`. |
| `UHRSWORK` | Continuous | Usual hours worked per week. |
| `TRANSPORT_METHOD`| Categorical | Broadened categories: `Private Vehicle`, `Public Transit`, `Active Transport` (Bike/Walk), `Worked at home`, `Other/Taxi`. |



### 5. Implementation Notes
1.  **Race/Ethnicity Priority:** Following standard demographic research practices, the `RACE_ETHNICITY` column prioritizes Hispanic identity over racial identity to create mutually exclusive categories.
2.  **Sparsity Management:** Columns such as `RACE` (codes 7, 8, 9) and `SPEAKENG` (Unknowns) were converted to `NaN` to ensure the LLM is evaluated on high-quality, high-frequency categories.
3.  **Missing Values:** All mapped categorical variables automatically return `NaN` for undefined codes (e.g., "N/A" or "Missing"), facilitating easy cleanup using `df.dropna()`.